In [0]:
# Ques3 Read the CSV file with header and automatically detect data types (inferSchema)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/week6_data/sales_data.csv")

In [0]:
# Display the schema to verify column names and data types
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
# Display the loaded dataset
df.show()

+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+
|product_id|   category|  price|base_price| amount|    status|region|priority|old_name|user_id|order_date|
+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+
|     P0001|Electronics| 222.55|     188.6|2822.79|   Pending| South|     Low|  Item_1|  81482|2025-02-14|
|     P0002|     Sports|2167.42|    1836.8| 394.99|   Pending| South|     Low|  Item_2|  83563|2025-04-12|
|     P0003|     Sports|2155.65|   1826.82|4547.17| Cancelled| North|    High|  Item_3|  54597|2025-05-23|
|     P0004|  Furniture|1155.04|    978.85|7658.59| Completed| North|  Medium|  Item_4|  55082|2025-11-06|
|     P0005|   Clothing|4054.93|   3436.38|7324.34| Completed|  West|    High|  Item_5|  92397|2025-11-13|
|     P0006|   Clothing|2929.03|   2482.23|7075.26| Completed| South|  Medium|  Item_6|  40512|2025-02-21|
|     P0007|      Books|1462.07|   12

In [0]:
# List the files available in the specified volume
display(dbutils.fs.ls("/Volumes/workspace/default/week6_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/week6_data/sales_data.csv,sales_data.csv,85507,1783228815000


In [0]:
# Ques5 Filter Electronics records and select product_id and price columns
electronics = df.filter(df.category == "Electronics") \
                .select("product_id", "price")
electronics.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|     P0001| 222.55|
|     P0008|3084.74|
|     P0010|1133.82|
|     P0014|3440.38|
|     P0015| 4773.7|
|     P0018|3686.35|
|     P0022|2619.36|
|     P0037|4459.59|
|     P0039|1734.43|
|     P0040| 648.18|
|     P0043| 978.87|
|     P0044|4307.11|
|     P0045|3334.16|
|     P0047| 3023.2|
|     P0054| 326.45|
|     P0056|2831.29|
|     P0064|4543.95|
|     P0065|1374.77|
|     P0067|3795.74|
|     P0068|2863.45|
+----------+-------+
only showing top 20 rows


In [0]:
# Ques 6 Rename the column and cast the price column to Double type
from pyspark.sql.functions import col

df = df.withColumnRenamed("old_name", "new_name") \
       .withColumn("price", col("price").cast("double"))

df.printSchema()
df.show()

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- order_date: date (nullable = true)

+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+
|product_id|   category|  price|base_price| amount|    status|region|priority|new_name|user_id|order_date|
+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+
|     P0001|Electronics| 222.55|     188.6|2822.79|   Pending| South|     Low|  Item_1|  81482|2025-02-14|
|     P0002|     Sports|2167.42|    1836.8| 394.99|   Pending| South|     Low|  Item_2|  83563|2025-04-12|
|     P0003|     Sports|

In [0]:
# Ques 8 Filter orders where status is Completed and amount is greater than 1000
completed = df.filter(
    (df.status == "Completed") &
    (df.amount > 1000)
)
completed.show()

+----------+-----------+-------+----------+-------+---------+------+--------+--------+-------+----------+
|product_id|   category|  price|base_price| amount|   status|region|priority|new_name|user_id|order_date|
+----------+-----------+-------+----------+-------+---------+------+--------+--------+-------+----------+
|     P0004|  Furniture|1155.04|    978.85|7658.59|Completed| North|  Medium|  Item_4|  55082|2025-11-06|
|     P0005|   Clothing|4054.93|   3436.38|7324.34|Completed|  West|    High|  Item_5|  92397|2025-11-13|
|     P0006|   Clothing|2929.03|   2482.23|7075.26|Completed| South|  Medium|  Item_6|  40512|2025-02-21|
|     P0009|     Sports|1176.11|     996.7| 3310.4|Completed| South|    High|  Item_9|  62581|2025-05-18|
|     P0012|     Sports|2518.23|   2134.09|7582.24|Completed| South|     Low| Item_12|  99192|2025-08-05|
|     P0020|   Clothing|2246.69|   1903.97| 5224.0|Completed| South|    High| Item_20|  12757|2025-10-29|
|     P0021|     Sports|1227.56|   1040.31|228

In [0]:
# Ques 10 Create a new final_price column by adding 18% tax
from pyspark.sql.types import DecimalType

df = df.withColumn(
    "final_price",
    (col("base_price") * 1.18
).cast(DecimalType(10,2))
)


df.show()

+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|product_id|   category|  price|base_price| amount|    status|region|priority|new_name|user_id|order_date|final_price|
+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|     P0001|Electronics| 222.55|     188.6|2822.79|   Pending| South|     Low|  Item_1|  81482|2025-02-14|     222.55|
|     P0002|     Sports|2167.42|    1836.8| 394.99|   Pending| South|     Low|  Item_2|  83563|2025-04-12|    2167.42|
|     P0003|     Sports|2155.65|   1826.82|4547.17| Cancelled| North|    High|  Item_3|  54597|2025-05-23|    2155.65|
|     P0004|  Furniture|1155.04|    978.85|7658.59| Completed| North|  Medium|  Item_4|  55082|2025-11-06|    1155.04|
|     P0005|   Clothing|4054.93|   3436.38|7324.34| Completed|  West|    High|  Item_5|  92397|2025-11-13|    4054.93|
|     P0006|   Clothing|2929.03|   2482.23|7075.

In [0]:
# Ques 12 Save the DataFrame as a Parquet file
df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/week6_data/parquet_data")

# Load the Parquet file
df_parquet = spark.read.parquet(
    "/Volumes/workspace/default/week6_data/parquet_data"
)

# Remove rows where user_id is null
filtered_df = df_parquet.filter(
    col("user_id").isNotNull()
)

filtered_df.show()

# Save the filtered data as a CSV file
filtered_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/week6_data/output_csv")

+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|product_id|   category|  price|base_price| amount|    status|region|priority|new_name|user_id|order_date|final_price|
+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|     P0001|Electronics| 222.55|     188.6|2822.79|   Pending| South|     Low|  Item_1|  81482|2025-02-14|     222.55|
|     P0002|     Sports|2167.42|    1836.8| 394.99|   Pending| South|     Low|  Item_2|  83563|2025-04-12|    2167.42|
|     P0003|     Sports|2155.65|   1826.82|4547.17| Cancelled| North|    High|  Item_3|  54597|2025-05-23|    2155.65|
|     P0004|  Furniture|1155.04|    978.85|7658.59| Completed| North|  Medium|  Item_4|  55082|2025-11-06|    1155.04|
|     P0005|   Clothing|4054.93|   3436.38|7324.34| Completed|  West|    High|  Item_5|  92397|2025-11-13|    4054.93|
|     P0006|   Clothing|2929.03|   2482.23|7075.

In [0]:
# Ques 14 Filter records where region is North or priority is High
result = filtered_df.filter(
    (filtered_df.region == "North") |
    (filtered_df.priority == "High")
)
result.show()

+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|product_id|   category|  price|base_price| amount|    status|region|priority|new_name|user_id|order_date|final_price|
+----------+-----------+-------+----------+-------+----------+------+--------+--------+-------+----------+-----------+
|     P0003|     Sports|2155.65|   1826.82|4547.17| Cancelled| North|    High|  Item_3|  54597|2025-05-23|    2155.65|
|     P0004|  Furniture|1155.04|    978.85|7658.59| Completed| North|  Medium|  Item_4|  55082|2025-11-06|    1155.04|
|     P0005|   Clothing|4054.93|   3436.38|7324.34| Completed|  West|    High|  Item_5|  92397|2025-11-13|    4054.93|
|     P0009|     Sports|1176.11|     996.7| 3310.4| Completed| South|    High|  Item_9|  62581|2025-05-18|    1176.11|
|     P0014|Electronics|3440.38|   2915.58|5416.01| Cancelled|  East|    High| Item_14|  30730|2025-08-21|    3440.38|
|     P0016|     Sports|3083.95|   2613.52|1613.

In [0]:
# We can verify that the Parquet and CSV output files have been created successfully
display(dbutils.fs.ls("/Volumes/workspace/default/week6_data/"))

display(dbutils.fs.ls("/Volumes/workspace/default/week6_data/parquet_data"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/week6_data/output_csv/,output_csv/,0,1783229740733
dbfs:/Volumes/workspace/default/week6_data/parquet_data/,parquet_data/,0,1783229740733
dbfs:/Volumes/workspace/default/week6_data/sales_data.csv,sales_data.csv,85507,1783228815000


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/week6_data/parquet_data/_SUCCESS,_SUCCESS,0,1783229738000
dbfs:/Volumes/workspace/default/week6_data/parquet_data/_committed_4530460866878125167,_committed_4530460866878125167,124,1783229738000
dbfs:/Volumes/workspace/default/week6_data/parquet_data/_started_4530460866878125167,_started_4530460866878125167,0,1783229738000
dbfs:/Volumes/workspace/default/week6_data/parquet_data/part-00000-tid-4530460866878125167-3f0d6a58-1bad-474e-b61c-066bfb78d442-441-1.c000.snappy.parquet,part-00000-tid-4530460866878125167-3f0d6a58-1bad-474e-b61c-066bfb78d442-441-1.c000.snappy.parquet,40330,1783229738000
